# Barter Python: Engine & Backtesting

This notebook covers the engine pipeline: loading configs, running backtests,
and the current status of strategy and risk management APIs.

## Topics Covered
- Loading `SystemConfig` from JSON and historical market data
- Running a backtest with `run_backtest()`
- `Balance` type for wallet representation
- Strategy API status (not yet exposed to Python)
- Risk management API status (not yet exposed to Python)


## Prerequisites

These notebooks assume `barter` is installed into the active IPython kernel, for example:

```bash
cd /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```

If you register a dedicated kernel, select that kernel before running the notebook.


In [ ]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root from the current working directory.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


In [ ]:
import json

import barter
from barter import Balance, run_backtest


---
## Part 1: Engine & Backtesting


## 1. Load Example Config and Historical Market Data

The repo already contains a `SystemConfig` example and matching historical trade events.

In [ ]:
config_path = REPO_ROOT / "barter" / "examples" / "config" / "backtest_config.json"
market_data_path = REPO_ROOT / "barter" / "examples" / "data" / "binance_spot_market_data_with_disconnect_events.json"

example_config = json.loads(config_path.read_text())
system_config_json = json.dumps(example_config["system"])
market_data_json = market_data_path.read_text()

print(config_path)
print(market_data_path)

## 2. Run the Backtest

The current Python API uses the built-in noop strategy and noop risk manager wired into `run_backtest()`. That makes this a baseline engine integration example rather than a strategy notebook.

In [ ]:
summary = await run_backtest(
    config_json=system_config_json,
    market_data_json=market_data_json,
    risk_free_return=example_config.get("risk_free_return", 0.05),
)

summary_dict = summary.to_dict()
print(summary)
print(sorted(summary_dict.keys()))

## 3. Inspect the Result

The summary payload is returned as a Python object with both `to_json()` and `to_dict()` helpers.

In [ ]:
print(summary.to_json()[:4000])

---
## Part 2: Strategy API Status


The Rust crate exposes rich strategy traits (`AlgoStrategy`,
`ClosePositionsStrategy`, `OnDisconnectStrategy`). The current Python
package does **not** expose custom strategy callbacks yet.
`run_backtest()` wires in the internal noop strategy so the engine can
execute, but strategy authoring still lives on the Rust side.


In [ ]:
public_api = sorted(name for name in dir(barter) if not name.startswith('_'))
public_api

## What Is Missing Relative to Rust

- No Python `AlgoStrategy` callback API
- No Python multi-strategy composition API
- No Python access to the internal engine-state snapshots used by Rust strategy traits

Those gaps are binding work, not notebook work, so this notebook stops at the currently supported engine boundary.

---
## Part 3: Risk Management API Status


The Rust engine supports custom `RiskManager` implementations. The
current Python package does not expose risk-manager callbacks yet;
`run_backtest()` uses the internal approve-all noop manager.


In [3]:
wallet = Balance(1000, 850)
print(wallet)
print({
    "total": wallet.total_f64,
    "free": wallet.free_f64,
    "locked": float(wallet.locked),
})

Balance(total=1000, free=850)
{'total': 1000.0, 'free': 850.0, 'locked': 150.0}


## What Is Missing Relative to Rust

- No Python callback for approving or refusing orders
- No exposed helpers for the Rust risk-check utility types
- No Python hook into the engine order-gating pipeline

Once those bindings exist, this notebook can become a true Python risk-manager tutorial rather than a status notebook.